# DX 704 Week 12 Project

This week's project will revisit the email spam classifier project from week 9 using large language model embeddings instead of custom features.


The full project description and a template notebook are available on GitHub: [Project 12 Materials](https://github.com/bu-cds-dx704/dx704-project-12).


## Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Part 1: Download Data Set

We will be using the Enron spam data set as prepared in this GitHub repository.

https://github.com/MWiechmann/enron_spam_data

You may need to download this differently depending on your environment.

In [1]:
!wget https://github.com/MWiechmann/enron_spam_data/raw/refs/heads/master/enron_spam_data.zip

--2026-11-20 22:42:25--  https://github.com/MWiechmann/enron_spam_data/raw/refs/heads/master/enron_spam_data.zip
Resolving github.com (github.com)... 140.82.114.3
Connecting to github.com (github.com)|140.82.114.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/MWiechmann/enron_spam_data/refs/heads/master/enron_spam_data.zip [following]
--2026-11-20 22:42:26--  https://raw.githubusercontent.com/MWiechmann/enron_spam_data/refs/heads/master/enron_spam_data.zip
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 2606:50c0:8002::154, 2606:50c0:8003::154, 2606:50c0:8001::154, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|2606:50c0:8002::154|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 15642124 (15M) [application/zip]
Saving to: ‘enron_spam_data.zip’

enron_spam_data.zip 100%[===================>]  14.92M  50.4MB/s    in 0.3s    

2026-11-20 22:42:26 (50.4 MB

In [2]:
import pandas as pd

In [3]:
# pandas can read the zip file directly
enron_spam_data = pd.read_csv("enron_spam_data.zip")
enron_spam_data

,Message ID,Subject,Message,Spam/Ham,Date
0,0,christmas tree farm pictures,NaN,ham,1999-12-10
1,1,"vastar resources , inc .","gary , production from the high island larger ...",ham,1999-12-13
2,2,calpine daily gas nomination,- calpine daily gas nomination 1 . doc,ham,1999-12-14
3,3,re : issue,fyi - see note below - already done .\nstella\...,ham,1999-12-14
4,4,meter 7268 nov allocation,fyi .\n- - - - - - - - - - - - - - - - - - - -...,ham,1999-12-14
...,...,...,...,...,...
33711,33711,= ? iso - 8859 - 1 ? q ? good _ news _ c = eda...,"hello , welcome to gigapharm onlinne shop .\np...",spam,2005-07-29
33712,33712,all prescript medicines are on special . to be...,i got it earlier than expected and it was wrap...,spam,2005-07-29
33713,33713,the next generation online pharmacy .,are you ready to rock on ? let the man in you ...,spam,2005-07-30
33714,33714,bloow in 5 - 10 times the time,learn how to last 5 - 10 times longer in\nbed ...,spam,2005-07-30


In [4]:
(enron_spam_data["Spam/Ham"] == "spam").mean()

0.5092834262664611

## Part 2: Download BERT Model

We will use a pre-trained BERT model to extract embedding vectors as described in lesson 2.1 this week.
Here is sample code to download the model from [Hugging Face](https://huggingface.co/) and extract one vector.
This model is small enough that you can run it with CPU only, but GPUs will be faster if available.

In [6]:
%pip install -U transformers torch

  Using cached huggingface_hub-2.4.0-py3-none-any.whl.metadata (16 kB)
  Using cached tokenizers-0.23.3-cp310-abi3-macosx_11_0_arm64.whl.metadata (9.8 kB)
  Using cached setuptools-84.2.1-py3-none-any.whl.metadata (6.6 kB)
  Using cached shellingham-1.5.4-py2.py3-none-any.whl.metadata (3.5 kB)
  Using cached rich-15.0.0-py3-none-any.whl.metadata (18 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 40.8 MB/s  0:00:00 eta 0:00:01
Using cached huggingface_hub-2.4.0-py3-none-any.whl (857 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 28.7 MB/s  0:00:00
Using cached tokenizers-0.23.3-cp310-abi3-macosx_11_0_arm64.whl (3.1 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.5/138.5 MB 62.5 MB/s  0:00:02 eta 0:00:01
Using cached setuptools-84.2.1-py3-none-any.whl (876 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 34.2 MB/s  0:00:00
Using cached rich-15.0.0-py3-none-any.whl (310 kB)
Using cached shellingham-1.5.4-py2.py3-none-any.whl (9.8 kB)
  Attempting un

In [7]:
# You may need to install torch and transformers.
# Google Colab will have these installed already.
#
# pip install transformers torch --upgrade

import torch
from transformers import AutoTokenizer, AutoModel

In [8]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cpu'

In [9]:
MODEL_NAME = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
bert_model = AutoModel.from_pretrained(MODEL_NAME)
bert_model.to(device)
bert_model.eval()


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

W1120 22:47:41.867000 73511 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
            (dropout): Dropout(p=

In [10]:
@torch.no_grad()
def embed_text(text):
    batch = [text]
    inputs = tokenizer(batch, padding=True, truncation=True, return_tensors="pt").to(device)
    outputs = bert_model(**inputs)
    # CLS token embedding is the first token's hidden state
    cls_emb = outputs.last_hidden_state[:, 0, :]  # shape: [batch_size, 768]
    return cls_emb.cpu()

In [11]:
v = embed_text("Hi, will you buy my spam?")
v.shape

torch.Size([1, 768])

## Part 3: Create Embedding Vectors

Use BERT to create embeddings for each email in the Enron data set.
You will have to decide how to combine the different columns of the original data set to produce one embedding vector.


Hint: BERT can be run without a GPU, but will be much slower.
Using Google Colab with only a CPU, it runs around 1 embedding per second.
Using Google Colab with the T4 GPU option, it runs around 60 embeddings per second.
Caching is also encouraged to avoid unnecessary reruns.

In [12]:
import numpy as np
from pathlib import Path

# Apple GPU is much faster than CPU here; falls back to the device chosen above.
device = "mps" if torch.backends.mps.is_available() else device
bert_model.to(device)

# One text per email: subject first so it survives BERT's 512-token truncation.
# Message is cut at 10,000 characters only to keep tokenizing fast; that is far beyond 512 tokens.
texts = (
    enron_spam_data["Subject"].fillna("")
    + ". "
    + enron_spam_data["Message"].fillna("").str.slice(0, 10000)
).tolist()


@torch.no_grad()
def embed_batch(batch):
    inputs = tokenizer(batch, padding=True, truncation=True, max_length=512, return_tensors="pt").to(device)
    outputs = bert_model(**inputs)
    return outputs.last_hidden_state[:, 0, :].cpu().numpy()


# Sorting by length keeps padding small; finished chunks are cached so an interrupted run can resume.
cache_dir = Path("embedding-cache")
cache_dir.mkdir(exist_ok=True)
order = np.argsort([len(t) for t in texts], kind="stable")
chunk_size, batch_size = 2048, 32

embeddings = np.zeros((len(texts), 768), dtype=np.float32)
for start in range(0, len(order), chunk_size):
    chunk_idx = order[start:start + chunk_size]
    chunk_path = cache_dir / f"chunk-{start:06d}.npy"
    if chunk_path.exists():
        chunk_embeddings = np.load(chunk_path)
    else:
        chunk_embeddings = np.concatenate([
            embed_batch([texts[i] for i in chunk_idx[b:b + batch_size]])
            for b in range(0, len(chunk_idx), batch_size)
        ])
        np.save(chunk_path, chunk_embeddings)
    embeddings[chunk_idx] = chunk_embeddings
    print(f"{min(start + chunk_size, len(order))}/{len(order)} embedded")

embeddings.shape

2048/33716 embedded
4096/33716 embedded
6144/33716 embedded
8192/33716 embedded
10240/33716 embedded
12288/33716 embedded
14336/33716 embedded
16384/33716 embedded
18432/33716 embedded
20480/33716 embedded
22528/33716 embedded
24576/33716 embedded
26624/33716 embedded
28672/33716 embedded
30720/33716 embedded
32768/33716 embedded
33716/33716 embedded


(33716, 768)

Save your embeddings in a file "embeddings.tsv.gz" with two columns, Message ID and embedding_vector_json, where embedding_vector_json is a JSON-encoded list.
Make sure that embedding_vector_json is a 1 dimensional list, not 2 dimensional.

Hint: don't forget the ".gz" extension indicating gzip compression.
The Pandas `.to_csv` method will automatically add the compression if you save data with a filename ending in ".gz", so you just need to pass it the right filename.

Hint: Gradescope only allows files up to 100MB to be submitted.
Round your embeddings to 3 decimal places to make them smaller.

In [13]:
import json

# Round in float64 so the JSON shows 0.123 rather than float32 noise like 0.12300000339.
rounded_embeddings = np.round(embeddings.astype(np.float64), 3)

embeddings_df = pd.DataFrame({
    "Message ID": enron_spam_data["Message ID"],
    "embedding_vector_json": [json.dumps(v.tolist()) for v in rounded_embeddings],
})
embeddings_df.to_csv("embeddings.tsv.gz", sep="\t", index=False)

Submit "embeddings.tsv.gz" in Gradescope.

## Part 4: Train a Linear Regression

Train an ordinary least squares regression for spam/ham status where spam is treated as target value 1 and ham is treated as target value 0 with your embeddings above as the only input variables.


In [14]:
import json

embedding_vectors = np.vstack([
    json.loads(v) for v in embeddings_df["embedding_vector_json"]
])
spam_targets = (enron_spam_data["Spam/Ham"] == "spam").astype(np.float64).to_numpy()

X = np.column_stack([np.ones(len(embedding_vectors)), embedding_vectors])
coefficients_with_bias = np.linalg.lstsq(X, spam_targets, rcond=None)[0]
bias = coefficients_with_bias[0]
embedding_coefficients = coefficients_with_bias[1:]

print("shape:", embedding_vectors.shape, "bias:", bias, "coefficients:", embedding_coefficients.shape)
print("training R^2:", 1 - np.sum((X @ coefficients_with_bias - spam_targets) ** 2) / np.sum((spam_targets - spam_targets.mean()) ** 2))

shape: (33716, 768) bias: 0.7362634567558665 coefficients: (768,)
training R^2: 0.8582349008189388


Save the coefficients of your linear model in a file "coefficients.tsv" with columns dim and coefficient where dim specifies the offset in the embedding vector (0-767).
Don't worry about the bias term (but your model should still have it).

In [15]:
coefficients_df = pd.DataFrame({
    "dim": np.arange(768),
    "coefficient": embedding_coefficients,
})
coefficients_df.to_csv("coefficients.tsv", sep="\t", index=False)

Submit "coefficients.tsv" in Gradescope.

## Part 5: Search for Relevant Documents

The file "queries.tsv" specifies 10 queries.
For each of the queries, encode them as a vector, and find the message that is closest using $L_2$.

In [16]:
queries_df = pd.read_csv("queries.tsv", sep="\t")

@torch.no_grad()
def embed_query_batch(query_texts):
    inputs = tokenizer(list(query_texts), padding=True, truncation=True, max_length=512, return_tensors="pt").to(device)
    outputs = bert_model(**inputs)
    return outputs.last_hidden_state[:, 0, :].cpu().numpy().astype(np.float64)

query_vectors = embed_query_batch(queries_df["query"])

# Use the saved, rounded email embeddings for the same 768 dimensions submitted in Part 3.
differences = embedding_vectors.astype(np.float64)[:, None, :] - query_vectors.astype(np.float64)[None, :, :]
nearest = np.argmin(np.sum(differences * differences, axis=2), axis=0)

print("query vectors:", query_vectors.shape)
queries_df[["query_id", "query"]].assign(nearest_row=nearest).assign(
    Message_ID=[enron_spam_data["Message ID"].iloc[i] for i in nearest]
)

query vectors: (10, 768)


,query_id,query,nearest_row,Message_ID
0,1,accounting arrangements,20071,20071
1,2,sales higher than production,14258,14258
2,3,asked lawyer to write letter about unexpected ...,18071,18071
3,4,engineering exam,13222,13222
4,5,discounted tickets,22253,22253
5,6,unexecuted agreement,4951,4951
6,7,well bore,4830,4830
7,8,capacity problem,4830,4830
8,9,london partner,4830,4830
9,10,dormant account,4951,4951


Save your results in a file "query-matches.tsv" with columns query_id, query_vector_json, and Message ID.

In [17]:
import json

query_matches_df = pd.DataFrame({
    "query_id": queries_df["query_id"],
    "query_vector_json": [json.dumps(v.tolist()) for v in query_vectors],
    "Message ID": [enron_spam_data["Message ID"].iloc[i] for i in nearest],
})
query_matches_df.to_csv("query-matches.tsv", sep="\t", index=False)

Submit "query-matches.tsv" in Gradescope.

## Part 6: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.
You do not need to provide code for data collection if you did that by manually.

## Part 7: Acknowledgements

If you discussed this assignment with anyone, please acknowledge them here.
If you did this assignment completely on your own, simply write none below.

If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for. If you did not use any other libraries, simply write none below.

If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy. If you did not use any generative AI tools, simply write none below.

### Acknowledgement

No other people provided assistance. I used pandas and NumPy for data handling and matrix calculations, PyTorch and Hugging Face Transformers for BERT embeddings, and scikit-learn only where not needed. GitHub Copilot Chat in Visual Studio Code assisted with environment setup, implementing and caching the BERT embedding process, fitting the least-squares model, saving all output files, and validating the results.

[Copilot Chat transcript](file:///Users/christopherabbas/Library/Application%20Support/Code/User/workspaceStorage/d690c5667068afdbf94ff162bc48cae2/GitHub.copilot-chat/transcripts/e7d90d9c-3572-4733-8668-e31c0026bd5e.jsonl)